In [ ]:
from hiceebox import GenomeView
from hiceebox.tracks import (
    HiCTriangleTrack,
    BigWigTrack,
    BedTrack,
    BedPETrack,
    GeneTrack,
)
from hiceebox.matrix import HicMatrixProvider, CoolerMatrixProvider
from hiceebox.utils.genomics import parse_region, format_region
try:
    from hiceebox.utils.genomics import region_from_gene_promoter, list_genes_in_promoter_bed
except ImportError:
    region_from_gene_promoter = None
    list_genes_in_promoter_bed = None  # Requires pip install -e .

import matplotlib.pyplot as plt
%matplotlib inline

print("HiCeeBox imported successfully!")
if region_from_gene_promoter is None:
    print("Note: Gene promoter positioning requires pip install -e . to have region_from_gene_promoter / list_genes_in_promoter_bed")

## Minimal runnable example (no Hi-C/BigWig files)

Below we create small example BED, BEDPE, and GTF files in memory so you can run a full plot without external data. For Hi-C and BigWig you would add your own files and tracks.

In [ ]:
import os
import tempfile

tmpdir = tempfile.mkdtemp(prefix='hiceebox_tutorial_')
print(f"Using temp dir: {tmpdir}")

# Example BED (same chromosome and range as our view)
bed_path = os.path.join(tmpdir, 'peaks.bed')
with open(bed_path, 'w') as f:
    f.write('chr6\t30100000\t30150000\n')
    f.write('chr6\t31100000\t31180000\n')
    f.write('chr6\t31500000\t31550000\n')

# Example BEDPE (two "loops" in region)
bedpe_path = os.path.join(tmpdir, 'loops.bedpe')
with open(bedpe_path, 'w') as f:
    f.write('chr6\t30100000\t30120000\tchr6\t31150000\t31170000\n')
    f.write('chr6\t30200000\t30250000\tchr6\t31400000\t31450000\n')

# Minimal GTF (one gene)
gtf_path = os.path.join(tmpdir, 'genes.gtf')
with open(gtf_path, 'w') as f:
    f.write('chr6\t.\texon\t30050000\t30070000\t.\t+\t.\tgene_id "G1"; gene_name "ExampleGene";\n')
    f.write('chr6\t.\texon\t30090000\t30110000\t.\t+\t.\tgene_id "G1"; gene_name "ExampleGene";\n')
    f.write('chr6\t.\texon\t30130000\t30160000\t.\t+\t.\tgene_id "G1"; gene_name "ExampleGene";\n')

print("Created example BED, BEDPE, and GTF files.")

In [ ]:
# Build a view with BED + BEDPE + Genes only (no Hi-C/BigWig)
view_demo = GenomeView(
    chrom='chr6',
    start=30_000_000,
    end=32_000_000,
    width=10,
    dpi=150,
    title='HiCeeBox demo: BED + BEDPE + Genes'
)

view_demo.add_track(BedTrack(bed_path, name='Peaks', color='red', style='box', height=0.5))
view_demo.add_track(BedPETrack(bedpe_path, name='Loops', style='arc', height=1.0))
view_demo.add_track(GeneTrack(gtf_path, name='Genes', show_gene_names=True, height=1.5))

fig = view_demo.plot(output=os.path.join(tmpdir, 'tutorial_out.pdf'), show=True, close=False)
print(f"Saved to {tmpdir}/tutorial_out.pdf")

## Plot and export

`view.plot()` draws all added tracks and can save to file and/or show interactively.

- **output**: path to file (`.pdf`, `.png`, or `.svg`)
- **show**: `True` to display (e.g. in Jupyter)
- **close**: `False` to keep the figure object for further use

In [ ]:
# Example usage:
# fig = view.plot(
#     output='figure.pdf',   # or .png, .svg
#     show=True,             # show in notebook
#     close=False            # return figure for further tweaks
# )

# Clear tracks and reuse the same view:
# view.clear_tracks()
# view.add_track(...)
# view.plot(...)

print("Use view.plot(output=..., show=..., close=...) and view.clear_tracks() to reuse the view.")

## Full example

- **Track height/scale**: Effective height = `height * scale`.
- **ATAC (BigWig) min/max**: `ylim=(min, max)` controls y-axis range (e.g., `ylim=(0, 10)`), `None` for auto; `show_ylim_labels=True` labels min/max on the left.
- **Gene**: `layout_mode='condensed'` single row compact, `'expanded'` multiple rows.
- **Region from gene**: If you have a promoter BED (e.g., `hg38.genecode.promoter.selected.2k5bp.bed`), use `region_from_gene_promoter('gene_name', 'promoter.bed', upstream=25000, downstream=25000)` to get `(chrom, start, end)` then create `GenomeView`. Two approaches are shown in the cell below.

In [ ]:
# Approach 1: Use gene promoter to locate region (requires promoter BED, name_column=4 means 4th column is gene name)
chrom, start, end = region_from_gene_promoter(
    'MYC',   # Change to your desired gene name; use list_genes_in_promoter_bed(promoter_bed_path) to see available genes
    '/Users/zchen6/Documents/Shenlab/zoechen0717/Test_data/hg38.genecode.promoter.selected.2k5bp.bed',
    upstream=500000, downstream=500000, name_column=6
)

# Approach 2: Direct coordinates
# chrom, start, end = 'chr6', 30_000_000, 32_000_000

view = GenomeView(chrom=chrom, start=start, end=end, width=8, dpi=300, title='My multi-omics view')

# Hi-C: vmin/vmax control colorbar range, cmap colormap, show_colorbar whether to show colorbar; other tracks use height/scale to adjust height
# Each track: height= relative height, scale= multiplier, effective height = height * scale
provider = HicMatrixProvider('/Users/zchen6/Documents/Shenlab/zoechen0717/Test_data/TP025-Arima-R1-filtered.hic')
view.add_track(HiCTriangleTrack(matrix_provider=provider, resolution=5000, norm='None', name='Hi-C', height=2.0, cmap='Reds', vmin=None, vmax=1, show_colorbar=False))

view.add_track(BigWigTrack('/Users/zchen6/Documents/Shenlab/zoechen0717/Test_data/Trt/TP017_S211_L003.100bp.fastp.R1.trim.srt.nodup.no_chrM_MT.tn5.pval.signal.bigwig', name='ATAC', color='blue', style='fill', height=0.5, ylim=(0, 100), show_ylim_labels=True))

view.add_track(BedTrack('/Users/zchen6/Documents/Shenlab/zoechen0717/Test_data/hg38.genecode.promoter.selected.2k5bp.bed', name='Peaks', color='red', height=0.1))
view.add_track(BedPETrack('/Users/zchen6/Documents/Shenlab/zoechen0717/Test_data/TP025_deep.100bp.fastp.10k.2.sig3Dinteractions.bedpe', name='Loops', style='arc', height=0.5, invert=True))  # Added invert=True for arcs downward

# Gene: max_genes=10 shows at most 10 genes; when using gene-based region, pass query_gene to ensure that gene is always displayed
view.add_track(GeneTrack('/Users/zchen6/Documents/Shenlab/zoechen0717/Test_data/gencode.v38.annotation.gtf.gz', name='Genes', show_gene_names=True, height=1, layout_mode='expanded', max_genes=10, query_gene='MYC'))  # query_gene matches the gene name used in region_from_gene_promoter above

view.plot(output='my_figure.pdf', show=True, close=False)